# Graph building blocks: examples

Runnable versions of the `PLAN.md` §3.3 examples that the current code supports. Everything runs on CPU with random data, so the numbers show the code works, not that a model is any good.

1. Wrap any model as a one-node graph
2. Train it with your own loop
3. Swap one part; nothing else changes
4. Deploy and predict on one image
5. Mistakes fail early, with the bad name
6. Record a run
7. Evaluate top-1 and top-5

Examples that need code which doesn't exist yet (`Trainer`, data modules, frozen teachers) are still in `PLAN.md` §3.3–3.4. Move each one here when its code lands.

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import tempfile

import torch
from torch import nn
from torchvision.models import resnet18

from lwfm.core.graph import ExecutionGraph, GraphError, ModelNode
from lwfm.core.losses import LossBinding, LossComposer
from lwfm.core.types import InputSpec
from lwfm.models.adapters import InputAdapter

_ = torch.manual_seed(0)

## 1. Wrap any model as a one-node graph

The model is an untouched torchvision `resnet18`. `reads` lists the values passed to `forward`, in order; `"inputs.image"` means `batch["inputs"]["image"]`. `writes` names what it returns; the node stores it under its own name, as `student.logits`. Preprocessing then the model is a straight chain, so it is plain torch: `nn.Sequential(InputAdapter(...), resnet18(...))`. The adapter resizes the 96×96 images to the 64×64 the model is set up for, then normalises them. The graph is only needed where values branch or merge.

In [3]:
student = ModelNode(
    name="student",
    module=nn.Sequential(  # a straight chain: plain torch
        InputAdapter(InputSpec(image_size=(64, 64))),
        resnet18(num_classes=10),  # untouched torchvision model
    ),
    reads="inputs.image",  # passed to forward as its first argument
    writes="logits",  # its Tensor return, stored as "student.logits"
)
graph = ExecutionGraph([student])

batch = {
    "inputs": {"image": torch.rand(8, 3, 96, 96)},  # resized to 64x64 by the adapter
    "targets": {"class": torch.randint(0, 10, (8,))},
    "meta": {"sample_id": [f"s{i}" for i in range(8)]},
}
out = graph(batch)
{name: tuple(value.shape) for name, value in out.items()}

{'student.logits': (8, 10)}

## 2. Train it with your own loop

The graph and the loss composer are enough to train; a `Trainer` will be a convenience, not a requirement. The loss reads `student.logits` and `targets.class`, passed to `CrossEntropyLoss(input, target)` in that order. Overfitting one batch is a quick check that gradients reach the weights: the loss should fall towards 0.

In [4]:
CE_READS = ["student.logits", "targets.class"]
loss_fn = LossComposer([LossBinding("ce", nn.CrossEntropyLoss(), CE_READS)])
optimizer = torch.optim.SGD(graph.parameters(), lr=0.1)

graph.train()
for step in range(20):  # overfit one batch: loss should fall
    total, terms = loss_fn(batch, graph(batch))
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    if step in (0, 19):
        print(step, {name: round(t.item(), 4) for name, t in terms.items()})

0 {'ce': 2.5103}


19 {'ce': 0.0012}


## 3. Swap one part; nothing else changes

**A different model.** `TinyViT` names its argument `pixels` and returns a dict. It is the last step of the chain, so its dict is what the node receives. Inputs are passed by position, so `reads` stays the same; `writes` lists the dict fields to keep. The loss still reads `student.logits`, so it needs no changes. The extra `cls_token` output becomes `student.cls_token`, ready for a later feature-distillation loss.

In [5]:
class TinyViT(nn.Module):
    def __init__(self, num_classes: int, dim: int = 32, patch: int = 8):
        super().__init__()
        self.embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=2 * dim, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=2, enable_nested_tensor=False
        )
        self.head = nn.Linear(dim, num_classes)

    def forward(self, pixels: torch.Tensor) -> dict[str, torch.Tensor]:
        tokens = self.embed(pixels).flatten(2).transpose(1, 2)  # [B, N, dim]
        tokens = torch.cat([self.cls.expand(len(tokens), -1, -1), tokens], dim=1)
        cls_token = self.encoder(tokens)[:, 0]
        return {"logits": self.head(cls_token), "cls_token": cls_token}


vit_student = ModelNode(
    name="student",
    module=nn.Sequential(
        InputAdapter(InputSpec(image_size=(64, 64))), TinyViT(num_classes=10)
    ),
    reads="inputs.image",
    writes=["logits", "cls_token"],  # dict fields to keep
)
vit_graph = ExecutionGraph([vit_student])
{name: tuple(value.shape) for name, value in vit_graph(batch).items()}

{'student.logits': (8, 10), 'student.cls_token': (8, 32)}

**A different loss, or a weighted mix.** $\mathcal{L} = 0.9\,\mathrm{CE}_{\text{smoothed}} + 0.1\,\mathrm{margin}$. Both are plain `torch.nn` losses, reading the same values. The composer returns the total for `backward()` and each term, detached, for logging.

In [6]:
mixed_loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(label_smoothing=0.1), CE_READS, weight=0.9
        ),
        LossBinding("margin", nn.MultiMarginLoss(), CE_READS, weight=0.1),
    ]
)
total, terms = mixed_loss(batch, vit_graph(batch))
total, terms

(tensor(2.5509, grad_fn=<AddBackward0>),
 {'ce': tensor(2.7049), 'margin': tensor(1.1647)})

## 4. Deploy and predict on one image

`deployable_model` returns the node's module, here the `nn.Sequential` of adapter and model, as a plain `nn.Module`. It contains no graph, names or losses. Its state dict holds only model weights (the adapter's mean and std are rebuilt from `InputSpec`), so a fresh copy reloads it and gives identical predictions. A random tensor stands in for an image file. For a real image: `read_image(path, mode=ImageReadMode.RGB).float() / 255` from `torchvision.io`.

In [7]:
model = graph.deployable_model("student").eval()  # InputAdapter + resnet18

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "model_weights.pt"
    torch.save(model.state_dict(), path)
    fresh = ExecutionGraph(
        [
            ModelNode(
                name="student",
                module=nn.Sequential(
                    InputAdapter(InputSpec(image_size=(64, 64))),
                    resnet18(num_classes=10),
                ),
                reads="inputs.image",
                writes="logits",
            )
        ]
    ).deployable_model("student")
    fresh.load_state_dict(torch.load(path, weights_only=True))
    fresh.eval()

image = torch.rand(3, 120, 80)  # [3, H, W] in [0, 1], any size
with torch.inference_mode():
    probs, classes = model(image[None]).softmax(-1).topk(5)
    torch.testing.assert_close(fresh(image[None]), model(image[None]))
classes, probs

(tensor([[7, 0, 3, 5, 6]]), tensor([[0.7956, 0.1326, 0.0320, 0.0271, 0.0037]]))

## 5. Mistakes fail early, with the bad name

The graph checks its wiring when it is built, before any data is touched: every value a node reads must come from the batch (`inputs.`, `targets.` or `meta.`) or from another node, node names must be unique, and there must be no cycles.

In [8]:
try:
    ExecutionGraph(
        [
            ModelNode("student", resnet18(), reads="inputs.image", writes="logits"),
            ModelNode(
                "head", nn.Linear(512, 10), reads="studnet.features", writes="logits"
            ),
        ]
    )
except GraphError as error:
    print(f"GraphError: {error}")

GraphError: 'head' reads 'studnet.features', which no node writes


## 6. Record a run

Training code **emits** events; **sinks** decide where they go. `LocalFileSink` writes the run directory: `manifest.json` (with versions and git state) and `status.json` straight away, each event to `events.jsonl` as it happens, and `summary.json` (the last value of each scalar) at the end. `ConsoleSink` prints one line per event. `OutputRouter` sends every event to both and, used with `with`, records the final status: `completed`, `failed` (with the traceback) or `interrupted`. Scalars can be passed as 0-dim tensors, as the loss composer returns them.

In [9]:
import json

from lwfm.results.events import RunEvent
from lwfm.results.router import OutputRouter
from lwfm.results.sinks import ConsoleSink, LocalFileSink

run_dir = Path(tempfile.mkdtemp()) / "demo_run"
sinks = [LocalFileSink(run_dir, manifest={"name": "notebook_demo"}), ConsoleSink()]

with OutputRouter(sinks) as router:
    graph.train()
    for step in range(3):
        total, terms = loss_fn(batch, graph(batch))
        optimizer.zero_grad()
        total.backward()
        optimizer.step()
        for name, term in terms.items():
            router.emit(
                RunEvent("scalar", f"train/loss/{name}", term, stage="fit", step=step)
            )

[fit step 0] train/loss/ce = 0.001193
[fit step 1] train/loss/ce = 0.001145
[fit step 2] train/loss/ce = 0.001102
status = completed


In [10]:
print(sorted(p.name for p in run_dir.iterdir()))
print("status:", json.loads((run_dir / "status.json").read_text())["status"])
print("events:", len((run_dir / "events.jsonl").read_text().splitlines()))
print("summary:", json.loads((run_dir / "summary.json").read_text()))

['events.jsonl', 'manifest.json', 'status.json', 'summary.json']
status: completed
events: 4
summary: {'train/loss/ce': 0.0011020149104297161}


## 7. Evaluate top-1 and top-5

A metric is an unchanged torchmetrics object; a `MetricBinding` names the values its `update(preds, target)` reads, in order. `Evaluator().evaluate` resets every metric, runs the graph in eval mode under `torch.inference_mode()`, updates each metric per batch and computes once at the end, so accuracy is over all samples (micro), not an average of batch averages. The graph is put back in train mode afterwards, so the same call works for validation during training. Any plain list or `DataLoader` of batches works as the loader. With an `event_sink`, values are emitted as `eval/<split>/<name>`. Both scores are 1.0 here only because the model overfit this very batch in section 2.

In [11]:
from torchmetrics.classification import MulticlassAccuracy

from lwfm.evaluation.evaluator import Evaluator
from lwfm.evaluation.metrics import MetricBinding

PRED_READS = ["student.logits", "targets.class"]  # update(preds, target)
metrics = [
    MetricBinding(
        f"top{k}",
        MulticlassAccuracy(num_classes=10, top_k=k, average="micro"),
        PRED_READS,
    )
    for k in (1, 5)
]
val_loader = [batch, batch]  # any iterable of batches; usually a DataLoader

result = Evaluator().evaluate(
    graph,
    val_loader,
    metrics,
    device=torch.device("cpu"),
    event_sink=OutputRouter([ConsoleSink()]),  # an EventSink
)
print(result.metrics, "| graph back in train mode:", graph.training)

eval/val/top1 = 1
eval/val/top5 = 1
{'top1': 1.0, 'top5': 1.0} | graph back in train mode: True
